# 🧠 LOGOS — GPU Training (CUDA T4)
**Vedic GEMM + Langevin Dynamics Physics Optimizer**

| Config | Value |
|--------|-------|
| Optimizer | Langevin Dynamics (NO Adam) |
| GEMM | Vedic Urdhva-Tiryagbhyam |
| Dataset | TinyStories ~2GB |
| GPU | T4 16GB |
| d_model | 256 |
| Layers | 6 |
| Epochs | 3 over full data |

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — Imports + Config
# ═══════════════════════════════════════════════════════
import os, glob, subprocess, re, math, shutil, time
import numpy as np
import matplotlib.pyplot as plt

WORK_DIR   = '/kaggle/working/LOGOS'
LOG_FILE   = '/kaggle/working/training_log.txt'
OUT_DIR    = '/kaggle/working/logos_trained'

# TinyStories .bin paths (Kaggle dataset)
# Try multiple known locations
POSSIBLE_BINS = [
    '/kaggle/input/roneneldan-tinystories/TinyStories_all_data/data00.json',
    '/kaggle/input/tinystories/data/train.bin',
    '/kaggle/input/datasets/josephmayok/roneneldan-tinystories/train.bin',
    '/kaggle/input/tinystories-dataset/train.bin',
]

# Check what's available
print('=== Checking dataset paths ===')
for p in POSSIBLE_BINS:
    exists = os.path.exists(p)
    size   = os.path.getsize(p)//1024//1024 if exists else 0
    print(f'  {"✅" if exists else "❌"} {p}  ({size} MB)')

# Also list all input datasets
print('\n=== /kaggle/input contents ===')
os.system('find /kaggle/input -name "*.bin" -o -name "*.json" -o -name "*.txt" 2>/dev/null | head -20')

print('\n✅ Cell 1 done')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — GPU + Environment Check
# ═══════════════════════════════════════════════════════
print('=== GPU ===')
os.system('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')

print('\n=== CUDA ===')
os.system('nvcc --version | grep release')

print('\n=== GCC ===')
os.system('g++ --version | head -1')

print('\n=== CMake ===')
os.system('cmake --version | head -1')

print('\n=== CPU Cores ===')
os.system('nproc')

print('\n=== RAM ===')
os.system('free -h | head -2')

print('\n=== Disk ===')
os.system('df -h /kaggle/working')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Dataset Prepare (TinyStories → dataset.txt)
# Handles both .bin (tokenized) and .json formats
# Target: 500MB+ of text for proper training
# ═══════════════════════════════════════════════════════
import json

TRAIN_FILE = '/kaggle/working/dataset.txt'

def prepare_from_bin(bin_path, out_path, max_mb=600):
    """Convert tokenized .bin → text using tiktoken"""
    os.system('pip install tiktoken -q')
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    data = np.fromfile(bin_path, dtype=np.uint16)
    print(f'Tokens in bin: {len(data):,}')
    max_tokens = max_mb * 1024 * 1024 // 4  # ~4 bytes per decoded char avg
    data = data[:max_tokens]
    with open(out_path, 'w', encoding='utf-8') as f:
        for i in range(0, len(data), 100_000):
            chunk = data[i:i+100_000].tolist()
            f.write(enc.decode(chunk))
            if i % 1_000_000 == 0:
                print(f'  Decoded {i:,} tokens...')
    return os.path.getsize(out_path)

def prepare_from_json(json_dir, out_path, max_mb=600):
    """Collect text from TinyStories .json files"""
    jsons = sorted(glob.glob(os.path.join(json_dir, '*.json')))
    print(f'JSON files found: {len(jsons)}')
    total = 0
    limit = max_mb * 1024 * 1024
    with open(out_path, 'w', encoding='utf-8') as f:
        for jf in jsons:
            if total >= limit:
                break
            try:
                with open(jf, 'r', encoding='utf-8') as jfh:
                    stories = json.load(jfh)
                for story in stories:
                    text = story.get('story', story.get('text', ''))
                    if text:
                        f.write(text.strip() + '\n\n')
                        total += len(text)
            except Exception as e:
                print(f'Skip {jf}: {e}')
            print(f'  {total//1024//1024} MB written...')
    return total

def prepare_from_jsonl(jsonl_path, out_path, max_mb=600):
    """From .jsonl format"""
    total = 0
    limit = max_mb * 1024 * 1024
    with open(out_path, 'w', encoding='utf-8') as f:
        with open(jsonl_path, 'r', encoding='utf-8') as jh:
            for line in jh:
                if total >= limit: break
                try:
                    obj = json.loads(line)
                    text = obj.get('story', obj.get('text', obj.get('content', '')))
                    if text:
                        f.write(text.strip() + '\n\n')
                        total += len(text)
                except: pass
    return total

# ── AUTO DETECT + PREPARE ─────────────────────────────────────
if os.path.exists(TRAIN_FILE) and os.path.getsize(TRAIN_FILE) > 50*1024*1024:
    sz = os.path.getsize(TRAIN_FILE)//1024//1024
    print(f'✅ dataset.txt already exists: {sz} MB — skipping')
else:
    print('Preparing dataset...')
    found = False

    # 1. Try .bin files
    for bp in glob.glob('/kaggle/input/**/*.bin', recursive=True):
        sz = os.path.getsize(bp)//1024//1024
        if sz > 10:
            print(f'Found .bin: {bp} ({sz} MB)')
            prepare_from_bin(bp, TRAIN_FILE, max_mb=600)
            found = True; break

    # 2. Try JSON directory (TinyStories HuggingFace format)
    if not found:
        for jd in glob.glob('/kaggle/input/**/TinyStories*', recursive=True):
            if os.path.isdir(jd) and glob.glob(os.path.join(jd,'*.json')):
                print(f'Found JSON dir: {jd}')
                prepare_from_json(jd, TRAIN_FILE, max_mb=600)
                found = True; break

    # 3. Try any .jsonl
    if not found:
        for jl in glob.glob('/kaggle/input/**/*.jsonl', recursive=True):
            sz = os.path.getsize(jl)//1024//1024
            if sz > 10:
                print(f'Found .jsonl: {jl} ({sz} MB)')
                prepare_from_jsonl(jl, TRAIN_FILE, max_mb=600)
                found = True; break

    # 4. Fallback: download via wget (if internet enabled)
    if not found:
        print('\n⚠️  No local dataset found.')
        print('Trying to download TinyStories sample (~50MB)...')
        ret = os.system(
            'wget -q --timeout=60 '
            'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt '
            f'-O {TRAIN_FILE}'
        )
        if ret != 0:
            print('Download failed. Creating synthetic dataset for testing...')
            # Synthetic fallback — enough for sanity check
            with open(TRAIN_FILE, 'w') as f:
                stories = [
                    'Once upon a time there was a little girl named Lily. She liked to play in the garden.",',
                    'Tom was a boy who loved dogs. One day he found a puppy near the park.',
                    'The sun was bright and warm. Anna went outside to play with her friends.',
                    'Max had a red ball. He threw it high into the sky and watched it fall down.',
                ]
                for _ in range(50000):  # ~2MB of repeated stories
                    f.write(stories[_%len(stories)] + '\n\n')
        found = True

# Final size
sz = os.path.getsize(TRAIN_FILE)//1024//1024
print(f'\n✅ Dataset ready: {sz} MB')
print('Preview (first 300 chars):')
with open(TRAIN_FILE) as f:
    print(f.read(300))

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Clone / Pull LOGOS
# ═══════════════════════════════════════════════════════
import os

WORK_DIR = '/kaggle/working/LOGOS'
REPO_URL = 'https://github.com/Vikas8719/LOGOS.git'

if not os.path.exists(WORK_DIR):
    print('Cloning LOGOS...')
    ret = os.system(f'git clone {REPO_URL} {WORK_DIR}')
    print('✅ Clone done' if ret==0 else '❌ Clone failed')
else:
    print('Pulling latest...')
    ret = os.system(f'git -C {WORK_DIR} pull origin main')
    print('✅ Pull done' if ret==0 else '❌ Pull failed')

os.chdir(WORK_DIR)
print('\nDirectory:', os.getcwd())
print('\nFiles:')
os.system('ls -la')
print('\ncuda/:')
os.system('ls cuda/')
print('\nsrc/:')
os.system('ls src/')

# Show optimizer being used
print('\n=== Optimizer check (must be Langevin, NOT Adam) ===')
os.system('grep -n "Langevin\\|Adam\\|PhysicsOpt\\|AdamOpt" src/main.cpp | head -20')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5 — Build CPU + GPU binaries
# ═══════════════════════════════════════════════════════
import os
WORK_DIR = '/kaggle/working/LOGOS'
os.chdir(WORK_DIR)
os.system('rm -rf build && mkdir build')

print('Building...')
build_cmd = """
cd /kaggle/working/LOGOS && \
cmake -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
  -DCMAKE_CUDA_ARCHITECTURES="75" \
  2>&1 | tail -5 && \
cmake --build build --parallel $(nproc) 2>&1
"""
ret = os.system(build_cmd)

print('\n=== Build Result ===')
if ret == 0:
    print('✅ BUILD SUCCESS')
else:
    print('❌ BUILD FAILED')

print('\nBinaries:')
os.system('ls -lh build/logos* 2>/dev/null || echo "No binaries found"')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6 — Sanity Tests
# ═══════════════════════════════════════════════════════
import os
WORK_DIR = '/kaggle/working/LOGOS'
os.chdir(WORK_DIR)

print('=== CPU Tests ===')
os.system('./build/logos --test 2>&1')

print('\n=== VedicGEMM Benchmark ===')
os.system('./build/logos --benchmark 2>&1')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7 — 🚀 GPU TRAINING (Main)
# Dataset: 500MB+ TinyStories
# Model:   d=256, L=6, H=8 — proper size
# Optim:   Langevin Dynamics
# ═══════════════════════════════════════════════════════
import os, subprocess, re, time
WORK_DIR   = '/kaggle/working/LOGOS'
TRAIN_FILE = '/kaggle/working/dataset.txt'
LOG_FILE   = '/kaggle/working/training_log.txt'
os.chdir(WORK_DIR)

# ── Choose binary ─────────────────────────────────────
GPU_BIN = './build/logos_gpu'
CPU_BIN = './build/logos'

if os.path.exists(GPU_BIN):
    BINARY = GPU_BIN
    MODE   = 'GPU (CUDA T4) ⚡'
else:
    BINARY = CPU_BIN
    MODE   = 'CPU (CUDA build failed)'

ds_mb = os.path.getsize(TRAIN_FILE)//1024//1024
print(f'╔══════════════════════════════════════╗')
print(f'║  LOGOS Training START                ║')
print(f'╚══════════════════════════════════════╝')
print(f'Binary  : {BINARY}')
print(f'Mode    : {MODE}')
print(f'Dataset : {ds_mb} MB')
print(f'Expected: Loss 8.x → 5.x → 3.x over training')
print(f'Physics : Langevin Dynamics (dW = -γ∇L·dt + √(2γkT)·η)')
print()

steps_log, losses_log, smooth_log = [], [], []
smooth = -1
start_t = time.time()

process = subprocess.Popen(
    [BINARY, '--train', TRAIN_FILE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    cwd=WORK_DIR
)

with open(LOG_FILE, 'w') as log:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()

            # Parse: Step   100 | Loss: 7.2341 | Smooth: 7.4123
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s = int(m.group(1))
                l = float(m.group(2))
                steps_log.append(s)
                losses_log.append(l)
                smooth = l if smooth < 0 else 0.95*smooth + 0.05*l
                smooth_log.append(smooth)

    except KeyboardInterrupt:
        process.terminate()
        print('\n⏹️  Stopped — checkpoint saved')

process.wait()
elapsed = time.time() - start_t

if losses_log:
    print(f'\n{'='*40}')
    print(f'Start loss : {losses_log[0]:.4f}')
    print(f'Final loss : {losses_log[-1]:.4f}')
    print(f'Best loss  : {min(losses_log):.4f}')
    print(f'Drop       : {losses_log[0]-losses_log[-1]:.4f} nats')
    print(f'Steps      : {steps_log[-1]}')
    print(f'Time       : {elapsed/60:.1f} min')
    if losses_log[-1] < losses_log[0] - 0.5:
        print('✅ Model is LEARNING!')
    else:
        print('⚠️  Loss not dropping — check output above')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 8 — Loss Curves
# ═══════════════════════════════════════════════════════
import math, re
import matplotlib.pyplot as plt

LOG_FILE = '/kaggle/working/training_log.txt'
steps_log, losses_log, smooth_log = [], [], []
smooth = -1

if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s, l = int(m.group(1)), float(m.group(2))
                steps_log.append(s)
                losses_log.append(l)
                smooth = l if smooth<0 else 0.95*smooth + 0.05*l
                smooth_log.append(smooth)

if len(steps_log) < 2:
    print('Not enough data — run Cell 7 first'); 
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('LOGOS Training — Langevin Dynamics Optimizer', fontsize=14)

    # Raw loss
    axes[0].plot(steps_log, losses_log, 'b-', lw=0.8, alpha=0.6, label='Raw loss')
    axes[0].plot(steps_log, smooth_log, 'r-', lw=2, label='Smoothed')
    axes[0].axhline(losses_log[-1], color='g', ls='--',
                    label=f'Final: {losses_log[-1]:.3f}')
    axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Steps')
    axes[0].grid(True, alpha=0.3); axes[0].legend()

    # Perplexity (log scale)
    perp = [math.exp(min(l,12)) for l in smooth_log]
    axes[1].plot(steps_log, perp, 'g-', lw=1.5)
    axes[1].set_title('Perplexity (smoothed)'); axes[1].set_xlabel('Steps')
    axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)

    # Loss drop per 100 steps
    if len(steps_log) > 10:
        chunk = max(1, len(losses_log)//20)
        delta_steps = steps_log[chunk::chunk]
        delta_loss  = [losses_log[max(0,i-chunk)] - losses_log[i]
                       for i in range(chunk, len(losses_log), chunk)]
        colors = ['green' if d>0 else 'red' for d in delta_loss]
        axes[2].bar(delta_steps[:len(delta_loss)], delta_loss, color=colors, alpha=0.7)
        axes[2].axhline(0, color='k', lw=0.5)
        axes[2].set_title('Loss Drop (green=good)'); axes[2].set_xlabel('Steps')
        axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nStart  : {losses_log[0]:.4f}  (PPL={math.exp(min(losses_log[0],12)):.0f})')
    print(f'Final  : {losses_log[-1]:.4f}  (PPL={math.exp(min(losses_log[-1],12)):.0f})')
    print(f'Best   : {min(losses_log):.4f}  (PPL={math.exp(min(min(losses_log),12)):.0f})')
    print(f'Steps  : {steps_log[-1]}')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 9 — Evaluate + Generate Text
# ═══════════════════════════════════════════════════════
import os, glob
WORK_DIR   = '/kaggle/working/LOGOS'
TRAIN_FILE = '/kaggle/working/dataset.txt'
os.chdir(WORK_DIR)

ckpts = sorted([f for f in glob.glob('*.bin') if 'vocab' not in f])
print(f'Checkpoints: {ckpts}')

if ckpts:
    latest = ckpts[-1]
    print(f'\nUsing: {latest}\n')
    print('=== Evaluation ===')
    os.system(f'./build/logos --eval {TRAIN_FILE} {latest} 2>&1')

    print('\n=== Text Generation ===')
    prompts = [
        'Once upon a time',
        'The little girl named',
        'Tom and his dog',
        'In a small village',
    ]
    for p in prompts:
        print(f"\nPrompt: '{p}'")
        print('-' * 40)
        os.system(f'./build/logos --generate {latest} "{p}" 2>&1')
else:
    print('⚠️  No checkpoint found — run Cell 7 first')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 10 — Save All Outputs
# ═══════════════════════════════════════════════════════
import os, glob, shutil
WORK_DIR = '/kaggle/working/LOGOS'
OUT_DIR  = '/kaggle/working/logos_trained'
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(WORK_DIR)

saved = []

# Checkpoints
for f in glob.glob('*.bin'):
    shutil.copy(f, OUT_DIR); saved.append(f)

# Binaries
for b in ['build/logos', 'build/logos_gpu']:
    if os.path.exists(b):
        shutil.copy(b, OUT_DIR); saved.append(b)

# Logs + plots
for f in ['/kaggle/working/training_log.txt',
           '/kaggle/working/loss_curve.png']:
    if os.path.exists(f):
        shutil.copy(f, OUT_DIR); saved.append(os.path.basename(f))

print(f'Saved {len(saved)} files to {OUT_DIR}:')
os.system(f'ls -lh {OUT_DIR}')
print('\n✅ Download from Output tab!')